# Perlin Noise

A self-contained refresher on **Perlin noise** — Ken Perlin's *gradient noise*, the workhorse
for generating **smooth, natural-looking pseudo-randomness**: terrain heightmaps, cloud and
marble textures, water ripples, fog, and any field that should vary *continuously* instead of
flickering like TV static. Invented for the 1982 film *Tron* (Perlin won a technical Oscar
for it) and refined in 2002 as **Improved Perlin Noise**.

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**Perlin noise is a function `noise(x, y)` that returns a smooth, repeatable, random-looking
value at every point in space.** Sample it on a grid and you get a height field that looks
like rolling hills; sample it in 3D and you get volumetric clouds or marble veins. The key
word is **smooth**: nearby inputs give nearby outputs (it's continuous and differentiable),
unlike `random()` which gives uncorrelated static.

**The problem it solves.** Plain `random()` per pixel is *white noise* — every sample is
independent, so the result is jagged, structureless TV snow. Real-world variation (mountains,
wood grain, wind) is **spatially coherent**: a point is similar to its neighbours but drifts
over distance. Perlin noise manufactures exactly that coherence cheaply and deterministically.

**Why "gradient" noise.** Perlin noise places a random *gradient vector* (a direction) at each
integer lattice point, and computes a value by smoothly blending the influence of the
surrounding corners' gradients. Because the value at every lattice point is forced to **zero**
and only the gradients differ, you never see the blocky "grid of values" artifact that simpler
*value noise* produces.

**Reach for it when:**
- You need **organic, continuous variation** — terrain, clouds, fire, water, ore veins, fog.
- You want it **deterministic & seekable**: `noise(x, y)` always returns the same value for the
  same input, so you can regenerate or stream an infinite world without storing it.
- You want to **stack scales** (octaves) into fractal detail — coarse hills + fine bumps.

**Look elsewhere when:**
- You need **discrete structure** — rooms, mazes, tile layouts (use BSP, WFC, cellular automata).
- You need **point distributions** (tree/enemy placement) — that's Poisson-disk or Worley.
- You're in 4D+ or want fewer directional artifacts and better speed — **Simplex noise** is the
  modern successor (see *When to Use vs Alternatives*).

## 2. Mental Model

**A field of tiny arrows on a grid, and you're reading how much each arrow "points toward" you.**

1. Lay down an integer **lattice**. At every corner, pin a random **unit gradient vector**
   (an arrow pointing some direction).
2. To evaluate `noise(x, y)`, find the square cell you're in. For each of the 4 corners,
   take the **dot product** of that corner's gradient with the vector *from the corner to your
   point*. Each dot product is a little ramp: positive where you're "downhill" in the arrow's
   direction, negative the other way, and **exactly zero at the corner itself**.
3. **Smoothly interpolate** those 4 ramp values using the fade curve
   `6t⁵ − 15t⁴ + 10t³` (an S-curve that's flat at both ends, so cell boundaries are seamless).

```
   g01•──────────•g11        Each corner has a random arrow (gradient g).
      │   .P     │            At point P, value = blend of 4 dot products:
      │          │              corner_grad · (P − corner)
   g00•──────────•g10        Fade-curve interpolation in x then y → smooth.
       value == 0 at every corner, so no blocky grid shows through.
```

The whole texture is the **sum of these per-cell blends**. Because corners are zero and the
fade curve kills the slope at boundaries, adjacent cells join seamlessly into one smooth field.
Add several copies at doubling frequencies (**octaves / fBm**) and you get fractal,
mountain-like detail.

## 3. Key Concepts

- **Lattice / grid** — integer points where gradients live. Cell size = one unit of "frequency".
- **Gradient vector** — a random unit direction pinned at each lattice point. Perlin noise =
  *gradient* noise because the random data is directions, not values.
- **Dot product ramp** — at each corner, `gradient · (point − corner)`. Linear, zero at the
  corner, signed by direction. The four corner ramps are what get blended.
- **Fade / ease curve** — `6t⁵ − 15t⁴ + 10t³` (Perlin's *quintic*). Its first **and second**
  derivatives are zero at 0 and 1, so interpolated cells meet with no visible seams or creasing.
  The original 1985 version used `3t² − 2t³` (smoothstep), which creases under lighting.
- **Lerp / bilinear blend** — interpolate the 4 corner ramps along x, then y (the fade curve
  warps the interpolation parameter first).
- **Frequency** — how many lattice cells per unit. Higher frequency = smaller features.
- **Amplitude** — vertical scale of one noise layer.
- **Octave** — one layer of noise at a given frequency/amplitude.
- **fBm (fractal Brownian motion)** — sum of octaves at **doubling frequency** and
  **halving amplitude**. This is what turns smooth blobs into realistic terrain.
- **Persistence** — amplitude multiplier per octave (≈0.5 typical). Lower = smoother; higher =
  rougher/noisier.
- **Lacunarity** — frequency multiplier per octave (≈2.0 typical, i.e. frequency doubles).
- **Permutation table** — the classic implementation hashes lattice coordinates through a
  shuffled 0–255 table (duplicated to 512) to pick each corner's gradient. Seeding = shuffling
  this table. (We use a vectorized random-angle variant below for clarity.)
- **Value range** — output is roughly `[−1, 1]` but **not guaranteed** to hit the extremes;
  normalize before mapping to colors/heights if you need a full [0,1].
- **Gradient vs value noise** — value noise interpolates random *values* at corners (blockier);
  gradient (Perlin) noise interpolates random *directions* (smoother, zero at lattice points).

## 4. Setup

Perlin noise is a handful of array operations — **NumPy alone** runs everything here, CPU-only,
in milliseconds. No GPU, no network, no API key.

```bash
%pip install numpy
# optional, only for a nicer image instead of the ASCII render:
%pip install matplotlib
```

Production code often reaches for a dedicated library instead of hand-rolling:

```bash
%pip install noise        # C-accelerated Perlin/Simplex (snoise2, pnoise2, ...)
%pip install perlin-numpy # vectorized NumPy Perlin + fBm
%pip install opensimplex  # patent-free Simplex successor
```

We implement it from scratch below so the mechanism is visible, then show the fractal (fBm)
layering. The optional matplotlib cell is **gated** so the notebook still runs top-to-bottom
without it.

In [1]:
import numpy as np

print("NumPy", np.__version__)

def perlin_2d(shape, res, seed=0):
    """Vectorized 2-D Perlin (gradient) noise.

    shape : (h, w) output pixels.  res : (ry, rx) lattice cells.
    `shape` must be an integer multiple of `res`. Returns ~[-1, 1].
    """
    rng = np.random.default_rng(seed)
    fade = lambda t: 6 * t**5 - 15 * t**4 + 10 * t**3   # Perlin quintic ease

    delta = (res[0] / shape[0], res[1] / shape[1])
    d = (shape[0] // res[0], shape[1] // res[1])
    # `grid` = fractional position (0..1) of every pixel within its lattice cell
    grid = np.mgrid[0:res[0]:delta[0], 0:res[1]:delta[1]].transpose(1, 2, 0) % 1

    # One random unit gradient vector per lattice corner
    angles = 2 * np.pi * rng.random((res[0] + 1, res[1] + 1))
    grad = np.dstack((np.cos(angles), np.sin(angles)))
    tile = lambda g: g.repeat(d[0], 0).repeat(d[1], 1)
    g00, g10 = tile(grad[:-1, :-1]), tile(grad[1:, :-1])
    g01, g11 = tile(grad[:-1, 1:]), tile(grad[1:, 1:])

    # Dot product of each corner's gradient with (pixel - corner)
    n00 = np.sum(np.dstack((grid[:, :, 0],     grid[:, :, 1]))     * g00, 2)
    n10 = np.sum(np.dstack((grid[:, :, 0] - 1, grid[:, :, 1]))     * g10, 2)
    n01 = np.sum(np.dstack((grid[:, :, 0],     grid[:, :, 1] - 1)) * g01, 2)
    n11 = np.sum(np.dstack((grid[:, :, 0] - 1, grid[:, :, 1] - 1)) * g11, 2)

    # Fade-curve interpolation: blend in x, then in y
    t = fade(grid)
    n0 = n00 * (1 - t[:, :, 0]) + t[:, :, 0] * n10
    n1 = n01 * (1 - t[:, :, 0]) + t[:, :, 0] * n11
    return np.sqrt(2) * ((1 - t[:, :, 1]) * n0 + t[:, :, 1] * n1)

field = perlin_2d((256, 256), (8, 8), seed=1)
print("output shape:", field.shape)
print("min %.3f  max %.3f  mean %.3f  std %.3f" %
      (field.min(), field.max(), field.mean(), field.std()))

NumPy 2.5.0
output shape: (256, 256)
min -0.881  max 0.757  mean 0.013  std 0.311


## 5. Worked Examples

### Example 1 — One octave of Perlin noise vs. white noise

The point of Perlin noise is **spatial coherence**. We render a single octave as ASCII
(dark `space` = low, bright `@` = high) and contrast it with `np.random` white noise of the
same size. Notice the Perlin field has smooth blobs and ridges; the white noise is structureless
static. A quick numeric proxy for "smoothness" is the **mean absolute difference between
neighbouring pixels** — much smaller for Perlin.

In [2]:
def ascii_render(a, w=56, h=22, title=""):
    chars = " .:-=+*#%@"
    h0, w0 = a.shape
    sub = a[:: max(1, h0 // h), :: max(1, w0 // w)][:h, :w]
    norm = (sub - sub.min()) / (np.ptp(sub) + 1e-9)
    if title:
        print(title)
    for row in norm:
        print("".join(chars[int(v * (len(chars) - 1))] for v in row))

def roughness(a):  # mean abs difference between horizontal neighbours
    return float(np.abs(np.diff(a, axis=1)).mean())

perlin = perlin_2d((256, 256), (8, 8), seed=3)
white = np.random.default_rng(3).standard_normal((256, 256))

ascii_render(perlin, title="Perlin noise (smooth, coherent):")
print("\nroughness  perlin = %.4f   white = %.4f   (lower = smoother)"
      % (roughness(perlin), roughness(white)))

Perlin noise (smooth, coherent):
====----=+*#%#*+=----=====-:::--==++======+++======--:--
++++====+*#%%%##*+======----=+**###*+=-::::::.......::-=
-=====++**##%###***++=++++**####*****++=--::.........:-=
=+++==--==++++====------==++*++==-==++++=--:--====-::::-
***+-:::--==-:....     ...:::------==+++=====+++++=---==
+=-::.:-=+++=-::..:::::::....::-=+++====++****+=----=++*
=-:. .:-=+****++=+++++++===-:::-=+++=---=+*###*+========
-:.    .:-=+*###******####*+-:.::-========++**++==--::::
=-:....::--+****+=----=+***+=-:::-=+***++==------::....:
=-:::-==+++****+=-:...:-==++====++**#*++=----=+++=-::::-
+++++*********++=-:   ..:-=++++++++***++=--=+*###*+==--=
++*#%##++====+++=-:.  .:-=+*#*+==---=++++===+***+=======
=+*###*+=---==++=-:. .:-=+*#%#*++=====+==---==+==---=+++
-===++=--:...:::::::::---==+***#******+++==---::::-=+*#*
=----===-::...::--=++++=--:::-=++*****+++++=-:::-=+*##*+
=---=++++=----=++**###**+=-::::--====----------=++*###*+
-::::-++****#############***++====--:...:::::--==+*###*

### Example 2 — Fractal noise (fBm): stacking octaves into terrain

A single octave is too smooth to look like real terrain. **fBm** sums several octaves, each at
**double the frequency and a fraction (`persistence`) of the amplitude**. Coarse octaves give
the big landforms; fine octaves add crinkle. Below we render 1, 3, and 5 octaves so you can see
detail accumulate, then map the 5-octave field through a "sea level" threshold to get a crude
land/water map.

In [3]:
def fbm_2d(shape, res, octaves=5, persistence=0.5, lacunarity=2, seed=0):
    out = np.zeros(shape)
    freq, amp, norm = 1, 1.0, 0.0
    for o in range(octaves):
        out += amp * perlin_2d(shape, (res[0] * freq, res[1] * freq), seed=seed + o)
        norm += amp
        freq *= lacunarity
        amp *= persistence
    return out / norm   # keep result in ~[-1, 1]

for oct_count in (1, 3, 5):
    f = fbm_2d((256, 256), (2, 2), octaves=oct_count, seed=7)
    ascii_render(f, h=12, title=f"--- fBm, {oct_count} octave(s) ---")
    print()

terrain = fbm_2d((256, 256), (2, 2), octaves=5, seed=7)
sea_level = 0.0
land_fraction = float((terrain > sea_level).mean())
print("5-octave terrain: min %.3f max %.3f | land above sea level = %.1f%%"
      % (terrain.min(), terrain.max(), 100 * land_fraction))

--- fBm, 1 octave(s) ---
-:::::......:::::----========-----:::::::::----===++++++
...       ....::---===+++++++++++============++++++*****
           ...::--==+++********************************+
       .....:::---===++++********##################***++
...::::::-------------======+++++***####%%%%%%%%%###**++
::---==========----:::::::::----==+++**###%%%%%%%%###**+
--====+++++====---:::........:::--==+++**##%%%%%%%%%###*
--====++++====---::.....   ....:::--==++**##%%%%%%%%%%%#
:::------------:::....        ....:::--==++***###%%%%%%#
  .....:::::::::::.......          ....:::--==+++****###
       ....:::::::--:::::::......        ...::---===++++
   .....:::----===========----:::...      ....:::--=====

--- fBm, 3 octave(s) ---
---:::-------------------:::---------=============--====
---------====---====+=================++++++++++======++
.::::::::::--==++****************+++****+++++++=========
.::----::::..::--==+++++======++++***************+++++++
.::::::::.......:::---------=====+++*

### Example 3 — Effect of `persistence` (and an optional image)

`persistence` controls how much each finer octave contributes. Low persistence → the coarse
octaves dominate → smooth, rolling hills. High persistence → fine octaves keep their punch →
rough, noisy, mountainous. We measure roughness across a sweep. The final cell renders a real
image **only if matplotlib is installed** (gated), so the notebook still runs without it.

In [4]:
for p in (0.25, 0.5, 0.75):
    f = fbm_2d((256, 256), (4, 4), octaves=6, persistence=p, seed=11)
    print("persistence %.2f -> roughness %.4f, std %.3f"
          % (p, roughness(f), f.std()))

# Optional: save a PNG only if matplotlib is available. Gated so a fresh
# kernel without matplotlib still executes this cell cleanly.
import importlib.util, os
if importlib.util.find_spec("matplotlib") and os.getenv("PERLIN_SAVE_PNG"):
    import matplotlib.pyplot as plt
    img = fbm_2d((256, 256), (4, 4), octaves=6, persistence=0.5, seed=11)
    plt.imshow(img, cmap="terrain")
    plt.axis("off")
    plt.savefig("perlin_terrain.png", bbox_inches="tight")
    print("saved perlin_terrain.png")
else:
    print("(matplotlib image skipped — set PERLIN_SAVE_PNG=1 with matplotlib "
          "installed to render a PNG; ASCII output above already shows the field)")

persistence 0.25 -> roughness 0.0085, std 0.236
persistence 0.50 -> roughness 0.0126, std 0.181
persistence 0.75 -> roughness 0.0299, std 0.139
(matplotlib image skipped — set PERLIN_SAVE_PNG=1 with matplotlib installed to render a PNG; ASCII output above already shows the field)


## 6. Gotchas & Pitfalls

- **Output range isn't a clean [-1, 1].** Perlin rarely reaches the theoretical extremes, and
  fBm tightens the range further. **Always normalize** (`(x - x.min()) / x.ptp()`) before
  mapping to colors, heights, or a [0,1] alpha.
- **Frequency vs. resolution confusion.** "More octaves" ≠ "higher resolution." Octaves add
  *detail at finer scales*; rendering more pixels just samples the same continuous function more
  densely. Increase `res`/frequency for smaller features, octaves for fractal richness.
- **Axis-aligned directional artifacts.** Classic Perlin noise has subtle bias along the grid
  axes (you can see faint horizontal/vertical streaks). This is *the* reason Perlin invented
  **Simplex noise** — it uses a simplex (triangular) grid that's far more isotropic.
- **Using the old `3t² − 2t³` fade.** The 1985 cubic smoothstep has a nonzero second derivative
  at cell edges, which shows up as creases under lighting/normal maps. Use the quintic
  `6t⁵ − 15t⁴ + 10t³` (done above).
- **Confusing Perlin with value noise.** Value noise interpolates random *values* and looks
  blockier; Perlin interpolates random *gradients* and is zero at lattice points. Many "Perlin"
  tutorials actually show value noise.
- **Tiling/seams.** Naive Perlin doesn't tile. To make it wrap, the gradients along opposite
  edges must match (wrap the permutation/lattice indices modulo the period), or sample noise on
  a torus in 4D. Don't expect seamless tiles for free.
- **Frequency must divide resolution (this implementation).** The vectorized `perlin_2d` above
  requires `shape` to be a multiple of `res`; otherwise `repeat` mismatches. Per-pixel
  permutation-table implementations don't have this constraint.
- **Seeding.** Reproducibility comes from the RNG/permutation seed. Re-seed deterministically
  per octave (we use `seed + o`) so re-runs match — but make sure octaves use *different* seeds,
  or they'd be scaled copies of the same pattern.
- **Simplex patent.** Ken Perlin patented *Simplex* noise (expired 2022); **OpenSimplex** was
  created as a patent-free alternative. Classic Perlin noise itself was never patent-encumbered.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-off vs Perlin |
|---|---|---|
| **Perlin noise** | Smooth continuous fields: terrain, clouds, textures, fog. | Axis-aligned artifacts; slower in high dimensions; not tileable for free. |
| **Simplex / OpenSimplex** | Same use cases, fewer directional artifacts, scales to 3D/4D cheaply. | Slightly more complex to implement; the modern default. See `noise-simplex-worley`. |
| **Value noise** | Quick-and-dirty smooth noise when you don't care about quality. | Blockier, more grid-aligned than gradient noise; cheaper. |
| **Worley / cellular noise** | Cell/blob patterns: cracks, scales, stone, water caustics, region maps. | Different *look* (Voronoi-like), not smooth hills. See `noise-simplex-worley`. |
| **Diamond-Square / midpoint displacement** | Fast fractal heightmaps on a fixed grid. | Tied to a grid, can show creasing; not a continuous `noise(x,y)`. See `diamond-square`. |
| **White noise (`random`)** | Grain, dithering, stochastic sampling. | No spatial coherence at all — the problem Perlin exists to solve. |
| **Poisson-disk / blue noise** | Even-but-random *point placement* (trees, stars, samples). | Produces points, not a continuous field. See `poisson-disk-sampling`. |

**Rule of thumb:** if you want a smooth, infinite, seekable scalar field that looks natural,
reach for gradient noise — and in new code prefer **Simplex/OpenSimplex** over classic Perlin
for fewer artifacts and better high-dimensional performance. Layer it with **fBm** for terrain,
threshold it for biomes/coastlines, and combine with **Worley** when you need cellular structure
on top.

## 8. Resources

- **Ken Perlin — "Improving Noise" (SIGGRAPH 2002, the Improved Perlin paper)** —
  https://mrl.cs.nyu.edu/~perlin/paper445.pdf
- **Ken Perlin — reference Java implementation of Improved Noise** —
  https://mrl.cs.nyu.edu/~perlin/noise/
- **Adrian Biagioli — "Understanding Perlin Noise" (the clearest walkthrough)** —
  https://adrianb.io/2014/08/09/perlinnoise.html
- **The Book of Shaders — chapters on Noise and Cellular Noise** —
  https://thebookofshaders.com/11/
- **Red Blob Games — "Making maps with noise functions"** —
  https://www.redblobgames.com/maps/terrain-from-noise/
- **`noise` (C-accelerated Perlin/Simplex for Python)** — https://pypi.org/project/noise/
- **`perlin-numpy` (vectorized NumPy Perlin + fBm, basis for the code here)** —
  https://github.com/pvigier/perlin-numpy
- Related notebooks in this domain: `noise-simplex-worley`, `diamond-square`,
  `poisson-disk-sampling`.